In [ ]:
# ================================================================
# EXAMPLE 7
# 2D TRANSIENT HEAT CONDUCTION USING EXPLICIT FDM (FTCS)
# Flexible Google Colab Version
# ================================================================
#
# Governing equation:
#
#       ∂T/∂t = α (∂²T/∂x² + ∂²T/∂y²)
#
# The program allows the user to change:
#
#       Lx, Ly
#       Nx, Ny
#       alpha
#       boundary temperatures
#       initial temperature
#       dt
#       t_final
#
# It also checks the FTCS stability condition automatically.
#
# ================================================================

import numpy as np
import matplotlib.pyplot as plt


# ================================================================
# 1. USER-DEFINED PARAMETERS
# ================================================================

# Plate dimensions
Lx = 1.0                  # Length in x-direction (m)
Ly = 1.0                  # Length in y-direction (m)

# Computational grid
Nx = 5                    # Number of nodes in x-direction
Ny = 5                    # Number of nodes in y-direction

# Material property
alpha = 1.0e-4            # Thermal diffusivity (m^2/s)

# Initial temperature
T_initial = 0.0           # Initial interior temperature (deg C)

# Boundary temperatures
T_left = 100.0              # Left boundary (deg C)
T_right = 100.0             # Right boundary (deg C)
T_bottom = 0.0            # Bottom boundary (deg C)

# T_top is applied only to the interior portion of the top boundary.
# The two top corner values are controlled by the left and right
# boundary conditions, as in Example 7.
T_top = 200.0             # Top interior boundary (deg C)

# Requested time step
dt_requested = 100      # s

# Requested final simulation time
t_final = 600.0           # s


# ================================================================
# 2. GRID
# ================================================================

dx = Lx / (Nx - 1)
dy = Ly / (Ny - 1)

x = np.linspace(0.0, Lx, Nx)
y = np.linspace(0.0, Ly, Ny)

X, Y = np.meshgrid(x, y)


print("================================================")
print("GRID INFORMATION")
print("================================================")

print(f"Lx = {Lx:.6g} m")
print(f"Ly = {Ly:.6g} m")
print(f"Nx = {Nx}")
print(f"Ny = {Ny}")
print(f"dx = {dx:.6g} m")
print(f"dy = {dy:.6g} m")
print(f"Total nodes    = {Nx * Ny}")
print(f"Interior nodes = {(Nx - 2) * (Ny - 2)}")

print()


# ================================================================
# 3. MAXIMUM STABLE TIME STEP
# ================================================================
#
# For the 2D explicit FTCS diffusion equation:
#
#       Fo_x + Fo_y <= 1/2
#
# where
#
#       Fo_x = alpha*dt/dx^2
#       Fo_y = alpha*dt/dy^2
#
# Therefore:
#
#       dt_max =
#
#       1
#       -----------------------------------------
#       2*alpha*(1/dx^2 + 1/dy^2)
#
# ================================================================

dt_max = 1.0 / (
    2.0
    * alpha
    * (
        1.0 / dx**2
        +
        1.0 / dy**2
    )
)


print("================================================")
print("STABILITY INFORMATION")
print("================================================")

print(f"Maximum stable dt = {dt_max:.6g} s")
print(f"Requested dt      = {dt_requested:.6g} s")


if dt_requested > dt_max:

    raise ValueError(
        "\nThe selected time step is unstable.\n"
        f"Choose dt <= {dt_max:.6g} s."
    )


# ================================================================
# 4. NUMBER OF TIME STEPS
# ================================================================
#
# ceil() is used so that the simulation reaches the requested
# final time.
#
# The effective time step is then slightly adjusted so that:
#
#       num_steps * dt = t_final
#
# exactly.
#
# Since dt <= dt_requested, stability is preserved.
#
# ================================================================

if t_final <= 0.0:

    raise ValueError("t_final must be greater than zero.")


num_steps = int(
    np.ceil(t_final / dt_requested)
)


dt = t_final / num_steps


Fo_x = alpha * dt / dx**2
Fo_y = alpha * dt / dy**2


print(f"Effective dt      = {dt:.6g} s")
print(f"Number of steps   = {num_steps}")

print()

print(f"Fo_x = {Fo_x:.6f}")
print(f"Fo_y = {Fo_y:.6f}")
print(f"Fo_x + Fo_y = {Fo_x + Fo_y:.6f}")


if Fo_x + Fo_y > 0.5 + 1.0e-12:

    raise ValueError(
        "FTCS stability condition violated."
    )

else:

    print("STABILITY CHECK: STABLE")


# ================================================================
# 5. BOUNDARY-CONDITION FUNCTION
# ================================================================

def apply_boundary_conditions(T):

    """
    Apply the prescribed Dirichlet boundary conditions.

    Array convention:

        T[0, :]   -> bottom boundary
        T[-1, :]  -> top boundary
        T[:, 0]   -> left boundary
        T[:, -1]  -> right boundary

    The top temperature is applied only to the interior portion
    of the top boundary. Therefore, the top two corner nodes
    retain the left- and right-boundary values.
    """

    # Bottom boundary
    T[0, :] = T_bottom

    # Top interior boundary
    T[-1, 1:-1] = T_top

    # Left boundary
    T[:, 0] = T_left

    # Right boundary
    T[:, -1] = T_right

    return T


# ================================================================
# 6. INITIAL TEMPERATURE FIELD
# ================================================================

T = np.full(
    (Ny, Nx),
    T_initial,
    dtype=float
)

T = apply_boundary_conditions(T)


print("\n================================================")
print("INITIAL TEMPERATURE FIELD")
print("================================================")

print(np.flipud(T))


# ================================================================
# 7. STORAGE
# ================================================================

T_history = [T.copy()]

time_history = [0.0]


# ================================================================
# 8. EXPLICIT FTCS TIME MARCHING
# ================================================================
#
# General update:
#
# T_P^(n+1)
#
# = T_P^n
#
# + Fo_x (T_E^n - 2T_P^n + T_W^n)
#
# + Fo_y (T_N^n - 2T_P^n + T_S^n)
#
# ================================================================

for n in range(num_steps):

    # Start from the old field
    T_new = T.copy()

    # Update interior nodes only
    for j in range(1, Ny - 1):

        for i in range(1, Nx - 1):

            T_new[j, i] = (

                T[j, i]

                + Fo_x * (
                    T[j, i + 1]
                    - 2.0 * T[j, i]
                    + T[j, i - 1]
                )

                + Fo_y * (
                    T[j + 1, i]
                    - 2.0 * T[j, i]
                    + T[j - 1, i]
                )
            )

    # Reapply fixed boundary conditions
    T_new = apply_boundary_conditions(T_new)

    # Advance to next time level
    T = T_new

    # Store results
    T_history.append(T.copy())

    time_history.append((n + 1) * dt)


# Convert lists to NumPy arrays
T_history = np.array(T_history)
time_history = np.array(time_history)


# ================================================================
# 9. PRINT TEMPERATURE FIELDS
# ================================================================

print("\n================================================")
print("TEMPERATURE FIELDS")
print("================================================")


for n in range(len(time_history)):

    print(
        f"\nt = {time_history[n]:.3f} s"
    )

    print(
        np.round(
            np.flipud(T_history[n]),
            4
        )
    )


# ================================================================
# 10. FINAL TEMPERATURE FIELD
# ================================================================

print("\n================================================")
print("FINAL TEMPERATURE FIELD")
print("================================================")

print(
    f"Final time = {time_history[-1]:.3f} s"
)

print(
    np.round(
        np.flipud(T_history[-1]),
        4
    )
)


# ================================================================
# 11. AUTOMATIC SELECTION OF FOUR PLOT TIMES
# ================================================================
#
# This avoids assuming that there are exactly four time levels.
#
# Four representative snapshots are selected from the complete
# simulation:
#
#       initial
#       approximately 1/3 of final time
#       approximately 2/3 of final time
#       final
#
# ================================================================

number_of_snapshots = min(
    4,
    len(time_history)
)


snapshot_indices = np.unique(

    np.linspace(
        0,
        len(time_history) - 1,
        number_of_snapshots,
        dtype=int
    )
)


# ================================================================
# 12. TEMPERATURE CONTOURS
# ================================================================

T_min = np.min(T_history)
T_max = np.max(T_history)

# Avoid identical contour limits in a trivial uniform-temperature case
if np.isclose(T_min, T_max):

    T_max = T_min + 1.0


levels = np.linspace(
    T_min,
    T_max,
    21
)


fig, axes = plt.subplots(
    2,
    2,
    figsize=(11, 9),
    constrained_layout=True
)


axes = np.asarray(axes).flatten()


# Turn off all axes initially.
# Only the required number will be used.
for ax in axes:

    ax.axis("off")


for plot_number, n in enumerate(snapshot_indices):

    ax = axes[plot_number]

    ax.axis("on")

    contour = ax.contourf(
        X,
        Y,
        T_history[n],
        levels=levels
    )

    # Display actual FDM nodes
    ax.scatter(
        X,
        Y,
        s=16
    )

    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")

    ax.set_title(
        f"t = {time_history[n]:.1f} s"
    )

    ax.set_aspect("equal")


fig.colorbar(
    contour,
    ax=axes.tolist(),
    shrink=0.88,
    label="Temperature T (deg C)"
)


fig.suptitle(
    "2D Transient Heat Conduction Using Explicit FTCS",
    fontsize=15
)


plt.show()